# 07 — Reddit Signal Collection

**Goal:** Collect r/Forex, r/worldnews, r/investing, r/Economics, r/stocks, and r/CryptoCurrency posts
from the Arctic Shift API (all post types) and persist them as Bronze-layer JSONL files for
downstream signal extraction.

**Strategy:**
- Explore-first: every conclusion is derived from observed cell output
- Incremental saves: each batch is written to disk immediately — no large in-memory accumulation
- One JSONL file per subreddit; the `subreddit` field inside each record identifies the source
- Do not filter post types at this stage — Bronze = raw and complete

## 1. Setup

In [ ]:
import requests
import json
import time
from pathlib import Path
from datetime import datetime, timezone

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT    = Path().resolve().parent
RAW_DIR = ROOT / "data" / "raw" / "reddit"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── Arctic Shift config ────────────────────────────────────────────────────────
BASE_URL    = "https://arctic-shift.photon-reddit.com/api/posts/search"
DATE_START  = "2021-01-01T00:00:00Z"
DATE_END    = "2025-12-31T23:59:59Z"
LIMIT       = 100
DELAY       = 0.5

SUBREDDITS = [
    "stocks",
]

print(f"Root      : {ROOT}")
print(f"Raw dir   : {RAW_DIR}")
print(f"Range     : {DATE_START}  →  {DATE_END}")
print(f"Subreddits: {SUBREDDITS}")

Root      : D:\SCRIPTS\FX-AlphaLab
Raw dir   : D:\SCRIPTS\FX-AlphaLab\data\raw\reddit
Range     : 2021-01-01T00:00:00Z  →  2025-12-31T23:59:59Z
Subreddits: ['investing', 'Economics', 'stocks']


## 2. API Probe

Fire a single request and inspect the raw response — field count, structure, what a real post looks like.

In [28]:
resp = requests.get(BASE_URL, params={
    "subreddit": "Forex",
    "after":     DATE_START,
    "before":    DATE_END,
    "limit":     3,
    "sort":      "asc",
}, timeout=30)

print(f"Status         : {resp.status_code}")
data = resp.json()
print(f"Top-level keys : {list(data.keys())}")
print(f"Posts returned : {len(data.get('data', []))}")

Status         : 200
Top-level keys : ['data']
Posts returned : 3


In [29]:
post = data["data"][0]
print(f"Total fields on first post: {len(post)}\n")
for k, v in sorted(post.items()):
    print(f"  {k:<35} {type(v).__name__:<10}  {str(v)[:80].replace(chr(10), ' ')}")

Total fields on first post: 80

  all_awardings                       list        []
  allow_live_comments                 bool        False
  archived                            bool        False
  author                              str         [deleted]
  author_created_utc                  NoneType    None
  author_flair_background_color       str         
  author_flair_css_class              NoneType    None
  author_flair_template_id            NoneType    None
  author_flair_text                   NoneType    None
  author_flair_text_color             str         dark
  can_gild                            bool        False
  category                            NoneType    None
  content_categories                  NoneType    None
  contest_mode                        bool        False
  created_utc                         int         1609459665
  discussion_type                     NoneType    None
  distinguished                       NoneType    None
  domain                

**Observations — API probe:**
*(fill in after running the cell above)*

## 3. Post Type Discovery

Probe for each expected post type to observe real field structures before writing any extraction logic.

In [30]:
def show_media_fields(post: dict) -> None:
    MEDIA_KEYS = [
        "post_hint", "is_video", "is_self", "is_gallery", "is_reddit_media_domain",
        "domain", "url", "url_overridden_by_dest",
        "media", "secure_media", "preview",
        "gallery_data", "media_metadata",
    ]
    for k in MEDIA_KEYS:
        present = k in post
        val = post.get(k)
        tag = "PRESENT" if present else "ABSENT "
        val_str = str(val)[:100].replace("\n", " ") if val not in (None, {}, []) else repr(val)
        print(f"  [{tag}] {k:<35} {val_str}")

print("Helpers ready.")

Helpers ready.


In [31]:
# Fetch a batch and classify by type
resp = requests.get(BASE_URL, params={
    "subreddit": "Forex",
    "after": DATE_START, "before": DATE_END,
    "limit": 100, "sort": "asc",
}, timeout=30)
sample = resp.json().get("data", [])

videos    = [p for p in sample if p.get("is_video")]
galleries = [p for p in sample if p.get("is_gallery")]
selfposts = [p for p in sample if p.get("is_self")]
images    = [p for p in sample if p.get("is_reddit_media_domain") and not p.get("is_video")]
external  = [p for p in sample if not p.get("is_self") and not p.get("is_video")
             and not p.get("is_gallery") and not p.get("is_reddit_media_domain")]

print(f"Sample size : {len(sample)}")
print(f"  Videos    : {len(videos)}")
print(f"  Galleries : {len(galleries)}")
print(f"  Self/text : {len(selfposts)}")
print(f"  Images    : {len(images)}")
print(f"  External  : {len(external)}")

Sample size : 100
  Videos    : 0
  Galleries : 1
  Self/text : 75
  Images    : 13
  External  : 11


In [32]:
# Inspect one post of each type present in the sample
type_map = {
    "VIDEO"    : videos,
    "GALLERY"  : galleries,
    "SELF/TEXT": selfposts,
    "IMAGE"    : images,
    "EXTERNAL" : external,
}
for label, bucket in type_map.items():
    print(f"\n{'='*60}")
    print(f"{label} ({len(bucket)} in sample)")
    print(f"{'='*60}")
    if bucket:
        show_media_fields(bucket[0])
    else:
        print("  (none in this sample)")


VIDEO (0 in sample)
  (none in this sample)

GALLERY (1 in sample)
  [ABSENT ] post_hint                           None
  [PRESENT] is_video                            False
  [PRESENT] is_self                             False
  [PRESENT] is_gallery                          True
  [PRESENT] is_reddit_media_domain              False
  [PRESENT] domain                              reddit.com
  [PRESENT] url                                 https://www.reddit.com/gallery/ko3ncr
  [PRESENT] url_overridden_by_dest              https://www.reddit.com/gallery/ko3ncr
  [PRESENT] media                               None
  [PRESENT] secure_media                        None
  [ABSENT ] preview                             None
  [PRESENT] gallery_data                        None
  [PRESENT] media_metadata                      None

SELF/TEXT (75 in sample)
  [ABSENT ] post_hint                           None
  [PRESENT] is_video                            False
  [PRESENT] is_self                

**Observations — Post types:**
*(fill in after running)*

## 4. Full Collection — Incremental

For each subreddit:
- Paginate from `DATE_START` to `DATE_END` using `created_utc` cursor
- Each batch is written to disk immediately as JSONL (one JSON object per line)
- No post objects are kept in memory after writing — only a `seen_ids` set (IDs only)
- Output: `data/raw/reddit/{subreddit.lower()}_raw_{RUN_TIMESTAMP}.jsonl`

**Run the collection cell manually. Do not re-run without checking for existing output files.**

In [33]:
RUN_TS = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")

def collect_subreddit(subreddit: str, after: str, before: str, out_path: Path, delay: float = 0.5) -> int:
    """
    Paginate through all posts for one subreddit, writing each batch to JSONL immediately.
    Returns number of posts written.
    """
    cursor:   str      = after
    batch_n:  int      = 0
    total:    int      = 0
    seen_ids: set[str] = set()

    with open(out_path, "a", encoding="utf-8") as f:
        while True:
            try:
                resp = requests.get(BASE_URL, params={
                    "subreddit": subreddit,
                    "after":     cursor,
                    "before":    before,
                    "limit":     LIMIT,
                    "sort":      "asc",
                }, timeout=30)
                resp.raise_for_status()
            except requests.RequestException as e:
                print(f"  [!] {e} — retrying in 5s")
                time.sleep(5)
                continue

            batch = resp.json().get("data", [])
            if not batch:
                print(f"  [r/{subreddit}] Empty batch — done.")
                break

            new = 0
            for post in batch:
                pid = post["id"]
                if pid not in seen_ids:
                    seen_ids.add(pid)
                    f.write(json.dumps(post, ensure_ascii=False) + "\n")
                    new += 1
            f.flush()

            total   += new
            batch_n += 1
            last_ts  = batch[-1]["created_utc"]
            last_dt  = datetime.fromtimestamp(last_ts, tz=timezone.utc).strftime("%Y-%m-%d")

            if batch_n % 50 == 0:
                print(f"  [r/{subreddit}] batch {batch_n:>5} | {total:>8,} posts | cursor {last_dt}")

            if len(batch) < LIMIT:
                print(f"  [r/{subreddit}] batch {batch_n:>5} | {total:>8,} posts | cursor {last_dt}  [DONE]")
                break

            cursor = datetime.fromtimestamp(last_ts + 1, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
            time.sleep(delay)

    return total


print(f"Run timestamp : {RUN_TS}")
print(f"Output files  :")
for sr in SUBREDDITS:
    p = RAW_DIR / f"{sr.lower()}_raw_{RUN_TS}.jsonl"
    print(f"  {p.name}")

Run timestamp : 20260405_230244
Output files  :
  investing_raw_20260405_230244.jsonl
  economics_raw_20260405_230244.jsonl
  stocks_raw_20260405_230244.jsonl


In [34]:
# ── RUN COLLECTION ─────────────────────────────────────────────────────────────
# Long-running (~30-90 min per large subreddit). Each subreddit writes to its
# own JSONL file and flushes to disk after every batch.
# Safe to interrupt — restart by pointing DATE_START to last printed cursor.
# ──────────────────────────────────────────────────────────────────────────────
summary = {}
t_total = time.time()

for subreddit in SUBREDDITS:
    out_path = RAW_DIR / f"{subreddit.lower()}_raw_{RUN_TS}.jsonl"
    print(f"\n{'─'*60}")
    print(f"Collecting r/{subreddit}  →  {out_path.name}")
    print(f"{'─'*60}")
    t0 = time.time()
    n  = collect_subreddit(subreddit, DATE_START, DATE_END, out_path)
    elapsed = time.time() - t0
    summary[subreddit] = {"posts": n, "file": str(out_path), "minutes": round(elapsed / 60, 1)}
    print(f"  → {n:,} posts  |  {elapsed/60:.1f} min")

print(f"\n{'='*60}")
print(f"COLLECTION COMPLETE — {(time.time()-t_total)/60:.1f} min total")
print(f"{'='*60}")
for sr, info in summary.items():
    print(f"  r/{sr:<20} {info['posts']:>9,} posts   {info['minutes']} min")


────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────


KeyboardInterrupt: 

## 5. Coverage Validation

Inspect the output files after collection — post counts, date coverage, file sizes, post type distribution per subreddit. Stream-reads line-by-line to stay memory-efficient.

In [ ]:
# Locate the most recent JSONL file per subreddit
files = {}
for sr in SUBREDDITS:
    matches = sorted(RAW_DIR.glob(f"{sr.lower()}_raw_*.jsonl"), reverse=True)
    if matches:
        files[sr] = matches[0]

print("Files found:")
for sr, p in files.items():
    size_mb = p.stat().st_size / 1_048_576
    print(f"  r/{sr:<20} {p.name}  ({size_mb:.1f} MB)")

: 

: 

: 

In [ ]:
# Post counts, date range, and type distribution — stream per file
from collections import Counter

for sr, path in files.items():
    dates, types = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = json.loads(line)
            dates.append(p["created_utc"])
            if p.get("is_video"):
                types.append("video")
            elif p.get("is_gallery"):
                types.append("gallery")
            elif p.get("is_self"):
                types.append("self")
            elif p.get("is_reddit_media_domain"):
                types.append("image")
            else:
                types.append("external")

    if not dates:
        print(f"r/{sr}: no data\n")
        continue

    min_dt = datetime.fromtimestamp(min(dates), tz=timezone.utc).strftime("%Y-%m-%d")
    max_dt = datetime.fromtimestamp(max(dates), tz=timezone.utc).strftime("%Y-%m-%d")
    counts = Counter(types)
    total  = sum(counts.values())

    print(f"r/{sr}")
    print(f"  Date range : {min_dt} → {max_dt}")
    print(f"  Total      : {total:,}")
    for t, n in counts.most_common():
        print(f"  {t:<12} {n:>8,}  ({n/total*100:.1f}%)")
    print()

: 

: 

: 

**Observations — Coverage:**
*(fill in after running)*